# Neural-ART compilation and deployment for NUCLEO-N657X0-Q

This notebook describes the conversion of a quantized TFLite model into the generated C interface and external-flash data required by the STM32N6 application. It also checks the resulting tensor interface, extracts the postprocessing constants, and verifies the programmed address span.

Reference mode (`USE_REFERENCE = True`) inspects the versioned COCO artifacts without invoking the model compiler. Custom mode (`False`) uses the TFLite model and class metadata produced by [Notebook 01](01_Train_and_Quantize_STM32N6.ipynb) and generates a deployment bundle in a separate working directory.

The compilation procedure uses ST Edge AI Core 3.0.0 to match the runtime included in this repository. The configuration cell accepts executable names available on PATH or absolute paths for `stedgeai` and `arm-none-eabi-objcopy`. Firmware build and programming procedures are documented in the [README](../README.md#5-build-the-application). The [architecture description](../Doc/Application-Overview.md) explains the relationship between camera buffers, generated network interfaces, and memory allocation.

In [ ]:
from pathlib import Path
import json
import math
import re
import shutil
import subprocess
import tempfile

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "Model").is_dir() else HERE.parent
# REPO = Path(r"C:\projects\STM32N6-Edge_Computer_Vision")
MODEL_DIR = REPO / "Model"
BOARD = "NUCLEO-N657X0-Q"
BOARD_DIR = MODEL_DIR / BOARD
assert (BOARD_DIR / "stai_network.h").is_file(), "Set REPO to the repository root."
WORK = REPO.parent / "stm32n6_model_work"

# Reference mode reads committed files. Custom mode runs tools and writes into a new staging directory.
USE_REFERENCE = True  # False enables compilation of the custom model.
MANIFEST = WORK / "insect_model.json"
TFLITE_OVERRIDE = None  # Optional: Path(r"C:\path\best_full_integer_quant.tflite")
STEDGEAI = "stedgeai"  # Or use the absolute path to stedgeai.exe.
OBJCOPY = "arm-none-eabi-objcopy"

## 1. Generated tensor interface

The YOLOv8 segmentation adapter consumes two signed 8-bit buffers in a fixed order. The first stores detection attributes before candidate indices. The second stores mask prototypes in height, width, channel order. The reference model produces 1,344 candidates with 116 attributes each and 32 prototype maps at 64 × 64 resolution. The attribute count comprises four box coordinates, 80 class scores, and 32 mask coefficients.

The generated `stai_network.h` defines the tensor interface used by the C application. Compilation can alter an interface dtype or its reported shape. In the reference model, ST replaces the signed TFLite input interface with an unsigned interface compatible with RGB888 camera bytes.

The parser below extracts shape, format, scale, and zero point from the generated macros. All of these properties are relevant to compatibility. Equal byte counts can describe different layouts or numerical encodings.

In [ ]:
def read_network_header(header_path):
    header = header_path.read_text(encoding="utf-8")
    # Joining C line continuations exposes each multiline macro as a single definition.
    header = header.replace(chr(92) + chr(10), " ")

    def macro(suffix):
        match = re.search(r"^#define[ 	]+STAI_NETWORK_" + suffix + r"[ 	]+(.+)$", header, re.M)
        assert match, f"Missing {suffix} in {header_path}"
        return match.group(1).strip()

    # Interface compatibility depends on both tensor layout and affine encoding.
    def spec(prefix):
        return {
            "format": macro(prefix + "_FORMAT").strip("()"),
            "shape": [int(x) for x in macro(prefix + "_SHAPE").strip("{} ").split(",")],
            "scale": float(macro(prefix + "_SCALES").strip("{} ")),
            "zero_point": int(macro(prefix + "_OFFSETS").strip("{} ")),
        }

    return {
        "input": [spec(f"IN_{i}") for i in range(1, int(macro("IN_NUM").strip("()")) + 1)],
        "outputs": [spec(f"OUT_{i}") for i in range(1, int(macro("OUT_NUM").strip("()")) + 1)],
    }

baseline_specs = read_network_header(BOARD_DIR / "stai_network.h")
print(json.dumps(baseline_specs, indent=2))

## 2. Model compilation and data packaging

The repository's generation script fixes the input filename to the bundled COCO model. This notebook retains the same compiler arguments and memory profile while accepting a configurable model path. A fresh staging directory preserves the profile's relative memory-pool path and isolates the output of each compilation attempt. [Repository generation script](../Model/generate-n6-model_NUCLEO-N657X0-Q.sh)

| Argument | Compilation or packaging function |
| --- | --- |
| `--target stm32n6` | Selects the STM32N6 code generator and runtime interface. |
| `default@user_neuralart_….json` | Loads the named compiler profile and its reference to the board memory pool. |
| `--input-data-type uint8` | Requests an unsigned input interface. Its generated scale and zero point determine whether raw RGB bytes can be supplied directly. |
| `--change-addresses 0x70380000` | Assigns the raw xSPI constants their destination address in the Intel HEX records. This address agrees with the compiler's external-flash pool. |

Reference mode reads the committed C/H files, model HEX, and generation report. Custom mode runs `stedgeai generate` and packages the generated xSPI data with `objcopy`. The resulting files comprise model-specific execution code, interface definitions, and constants for external NOR. The equivalent shell procedure is documented in the [README](../README.md#3-generate-the-model-files).

The generation report describes network allocations and execution placement. The firmware linker map describes application allocations. Both are required to assess memory compatibility and identify CPU fallback operations.

In [ ]:
PROFILE = f"user_neuralart_{BOARD}.json"
MPOOL = f"stm32n6-app2_{BOARD}.mpool"
GENERATED_C = ("network.c", "network_ecblobs.h", "stai_network.c", "stai_network.h")

def run_checked(args, cwd):
    # Captured diagnostics remain visible in Jupyter. A nonzero tool exit aborts the procedure.
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True, errors="replace")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

if USE_REFERENCE:
    GENERATED = BOARD_DIR
    NETWORK_HEX = BOARD_DIR / "network_data.hex"
    REPORT = MODEL_DIR / "st_ai_output" / "network_generate_report.txt"
    class_names = None
else:
    metadata = json.loads(MANIFEST.read_text(encoding="utf-8"))
    class_names = metadata["class_names"]
    assert class_names and len(class_names) <= 255 and metadata["imgsz"] == 256
    tflite = Path(TFLITE_OVERRIDE or metadata["model"]).resolve()
    assert tflite.is_file(), tflite

    WORK.mkdir(exist_ok=True)
    # A unique staging directory isolates generated files from earlier attempts.
    stage = Path(tempfile.mkdtemp(prefix="nucleo_", dir=WORK))
    (stage / "my_mpools").mkdir()
    # The JSON resolves its memory pool relative to this working directory.
    shutil.copy2(MODEL_DIR / PROFILE, stage)
    shutil.copy2(MODEL_DIR / "my_mpools" / MPOOL, stage / "my_mpools")
    run_checked([
        STEDGEAI, "generate", "--model", str(tflite), "--target", "stm32n6",
        "--st-neural-art", f"default@{PROFILE}", "--input-data-type", "uint8",
    ], stage)
    GENERATED = stage / "st_ai_output"
    REPORT = GENERATED / "network_generate_report.txt"
    raw = GENERATED / "network_atonbuf.xSPI2.raw"
    assert raw.is_file() and raw.stat().st_size > 0, raw
    # Raw constants have no address. Intel HEX includes the address used by CubeProgrammer.
    NETWORK_HEX = stage / "network_data.hex"
    run_checked([
        OBJCOPY, "-I", "binary", str(raw), "--change-addresses",
        "0x70380000", "-O", "ihex", str(NETWORK_HEX),
    ], stage)

for name in GENERATED_C:
    assert (GENERATED / name).is_file(), GENERATED / name
assert NETWORK_HEX.is_file() and REPORT.is_file()
print("Generated code:", GENERATED)
print("Model data:", NETWORK_HEX)
print("Report:", REPORT)

## 3. Interface verification and postprocessing parameters

The following checks establish the expected input format, dimensions, affine encoding, and output order. The firmware supplies RGB bytes directly, which requires the reference unsigned input encoding with scale approximately 1/255 and zero point 0. The decoder expects normalized box centers and dimensions, class scores, and mask coefficients in the documented layouts.

Detection and prototype tensors have independent scales and zero points. The reference detection output uses zero point 25, while the prototype output uses −115. The C adapter reads both pairs from `app_config.h`, so these definitions require synchronization with the newly generated header.

The cell prints replacement definitions for the class table and quantization pairs and extracts memory and execution information from the report. These structural checks complement evaluation on known images, which establishes whether the decoder interprets the exported values correctly.

In [ ]:
result_specs = read_network_header(GENERATED / "stai_network.h")
input_specs = result_specs["input"]
output_specs = result_specs["outputs"]
print(json.dumps(result_specs, indent=2))

assert len(input_specs) == 1 and len(output_specs) == 2
# The camera writes bytes directly. Scale 1/255 and zero point 0 make those bytes represent normalized RGB.
inp = input_specs[0]
assert inp["format"] == "STAI_FORMAT_U8" and inp["shape"] == [1, 256, 256, 3]
assert inp["zero_point"] == 0 and math.isclose(inp["scale"], 1 / 255, rel_tol=1e-5), (
    "The model input cannot use raw RGB bytes directly. Adjust preprocessing."
)
# The adapter interprets output 0 as detections and output 1 as prototypes.
detections, prototypes = output_specs
class_count = 80 if USE_REFERENCE else len(class_names)
assert detections["shape"] == [4 + class_count + 32, 1344, 1]
assert prototypes["shape"] == [1, 64, 64, 32]
assert all(s["format"] == "STAI_FORMAT_S8" and s["scale"] > 0 for s in output_specs)

if class_names is not None:
    # Generated definitions correspond to the configuration fields in Inc/app_config.h.
    print(f"#define NB_CLASSES ({class_count})")
    label_literals = [json.dumps(name, ensure_ascii=False) for name in class_names]
    labels = ", ".join(label_literals)
    print("#define CLASSES_TABLE", "const char* classes_table[NB_CLASSES] = {" + labels + "};")
print(f'#define AI_YOLOV8_SEG_ZERO_POINT ({detections["zero_point"]})')
print(f'#define AI_YOLOV8_SEG_SCALE ({detections["scale"]:.9f}f)')
print(f'#define AI_YOLOV8_SEG_MASK_ZERO_POINT ({prototypes["zero_point"]})')
print(f'#define AI_YOLOV8_SEG_MASK_SCALE ({prototypes["scale"]:.9f}f)')

# Report entries identify actual allocations and CPU fallback operations.
print("\nMemory usage and execution placement:")
for line in REPORT.read_text(encoding="utf-8", errors="replace").splitlines():
    if any(key in line for key in ("used)", "Total number of epochs", "pure software", "hybrid epochs", "pure hardware")):
        print(line)

### 3.1. External-flash address verification

Intel HEX represents data as records containing a byte count, address, record type, payload, and checksum. For the linear-address files generated here, type 04 records specify the upper 16 address bits. Type 00 records provide the lower 16-bit offset and the data bytes.

The parser checks record length and checksum, then calculates the occupied address span. The first data address is `0x70380000`. The final byte must fit within the NUCLEO's 64 MiB NOR, ending at `0x73FFFFFF`. The physical capacity provides an additional constraint to the memory-pool allocation used by the model compiler.

In [ ]:
first_address = None
end_address = 0  # Exclusive bounds make the byte span a simple subtraction.
upper_address = 0

with NETWORK_HEX.open(encoding="ascii") as hex_file:
    for line in hex_file:
        line = line.strip()
        assert line.startswith(":"), "Expected an Intel HEX record."
        record = bytes.fromhex(line[1:])
        count = record[0]
        assert len(record) == count + 5 and sum(record) % 256 == 0, "Invalid HEX record."
        offset = int.from_bytes(record[1:3], "big")
        record_type = record[3]
        if record_type == 4:
            upper_address = int.from_bytes(record[4:6], "big") << 16
        elif record_type == 0 and count:
            start = upper_address + offset
            first_address = start if first_address is None else min(first_address, start)
            end_address = max(end_address, start + count)

assert first_address == 0x70380000, f"Unexpected weights base: {first_address!r}"
assert end_address <= 0x74000000, "Model data extends past the physical external flash."
print(f"Model data spans {first_address:#010x} to {end_address - 1:#010x} "
      f"({(end_address - first_address) / 1024**2:.3f} MiB address span).")

## 4. Firmware integration and board validation

Deployment requires the generated network code and external constants to originate from the same compilation. The integration sequence is as follows.

1. Copy `network.c`, `network_ecblobs.h`, `stai_network.c`, `stai_network.h`, and `network_data.hex` into `Model/NUCLEO-N657X0-Q/`. The intermediate xSPI data can also be retained as `network_data.xSPI2.bin`.
2. Update the corresponding definitions in `Application/NUCLEO-N657X0-Q/Inc/app_config.h` using the verified class order and output quantization. The existing `POSTPROCESS_ISEG_YOLO_V8_UI` adapter applies to models with the same decoding semantics.
3. Review network memory allocations and fallback operations, then clean and rebuild the UVCL application. The [build procedure](../README.md#5-build-the-application) documents CubeIDE and Makefile image preparation.
4. Program the application at `0x70100000` and the matching model HEX at its encoded base of `0x70380000`. A fresh installation also requires `FSBL/ai_fsbl.hex` at `0x70000000`. Loader selection and boot configuration are documented in the [programming procedure](../README.md#6-program-external-flash).

Validation records the UART model identity, predictions on positive and empty scenes, memory usage, and latency. A numerical PC-to-board comparison uses identical RGB input bytes and examines raw tensors before postprocessing. The demo timer covers the network wrapper. Capture, decoding, and display timing require separate measurements for a complete latency assessment.

In a ToF-triggered imaging application, inference follows the acquisition trigger. Predictions that control a high-resolution crop require conversion from model coordinates to original-frame coordinates using the inverse preprocessing transform.